Students implement Backtracking Search to solve the graph coloring problem. \
Students can add supporting attributes and methods to related classes as needed.

# Libraries

In [1]:
import os
import heapq

# Graph class

In [8]:
# Directed, weighted graphs
class Graph:
  def __init__(self):
    self.AL = dict() # adjacency list
    self.V = 0
    self.E = 0

  def __str__(self):
    res = 'V: %d, E: %d\n'%(self.V, self.E)
    for u, neighbors in self.AL.items():
      line = '%d: %s\n'%(u, str(neighbors))
      res += line
    return res

  def print(self):
    print(str(self))

  def load_from_file(self, filename):
    '''
        Example input file:
            V E
            u v w
            u v w
            u v w
            ...
            u1 h1
            u2 h2
            u3 h3
            ...

        # input.txt
        7 8
        0 1 1
        0	2 1
        1	2 1
        1	3 1
        2	4 1
        3	4 1
        4	5 1
        5	6 1
    '''
    if os.path.exists(filename):
      with open(filename) as g:
        self.V, self.E = [int(it) for it in g.readline().split()]
        for i in range(self.E):
          line = g.readline()
          u, v, w = [int(it) for it in line.strip().split()]
          if u not in self.AL:
            self.AL[u] = []
          self.AL[u].append((v, w))

In [9]:
g = Graph()
g.load_from_file('input.txt')
g.print()

V: 7, E: 8
0: [(1, 1), (2, 1)]
1: [(2, 1), (3, 1)]
2: [(4, 1)]
3: [(4, 1)]
4: [(5, 1)]
5: [(6, 1)]



# Search Strategies

In [10]:
class BackTrackingSearch:
  def search(self, csp: Graph) -> dict:
    '''
    return a dict {node : color} in which
      node: a node in the given graph
      color: the assigned color, {red, green, blue}
    '''
    return self.BACKTRACKING(dict(), csp)

  def BACKTRACKING(self, assignment: dict, csp: Graph) -> dict:
    """
    Backtracking search implementation following the pseudocode
    """
    if len(assignment) == csp.V:
      return assignment

    var = self.SELECT_UNASSIGNED_VARIABLE(csp, assignment)

    for value in self.ORDER_DOMAIN_VALUES(var, assignment, csp):
      if self.is_consistent(var, value, assignment, csp):
        assignment[var] = value

        inferences = self.INFERENCE(csp, assignment, var, value)

        if inferences is not None:
          result = self.BACKTRACKING(assignment, csp)
          if result is not None:
            return result

        del assignment[var]

    return None

  def SELECT_UNASSIGNED_VARIABLE(self, csp: Graph, assignment: dict) -> int:
    """
    Select an unassigned variable (node)
    """
    for node in range(csp.V):
      if node not in assignment:
        return node
    return None

  def ORDER_DOMAIN_VALUES(self, var: int, assignment: dict, csp: Graph) -> list:
    """
    Return possible color values
    """
    return ['red', 'green', 'blue']

  def is_consistent(self, var: int, value: str, assignment: dict, csp: Graph) -> bool:
    """
    Check if the color assignment is consistent with neighbors.
    Checks outgoing edges (AL) AND scans assigned nodes for any
    incoming edge pointing to var — without modifying Graph.
    """
    # Check outgoing neighbors (var -> neighbor)
    if var in csp.AL:
      for (neighbor, _) in csp.AL[var]:
        if neighbor in assignment and assignment[neighbor] == value:
          return False
    # Check incoming neighbors (assigned node -> var)
    for assigned_node in assignment:
      if assigned_node in csp.AL:
        for (neighbor, _) in csp.AL[assigned_node]:
          if neighbor == var and assignment[assigned_node] == value:
            return False
    return True

  def INFERENCE(self, csp: Graph, assignment: dict, var: int, value: str) -> dict:
    """
    Perform constraint propagation.
    Returns None if inference leads to a failure.
    """
    if var in csp.AL:
      for (neighbor, _) in csp.AL[var]:
        if neighbor not in assignment and not self.has_valid_color(neighbor, value, assignment, csp):
          return None
    return {}

  def has_valid_color(self, node: int, excluded_color: str, assignment: dict, csp: Graph) -> bool:
    """
    Check if the node has any valid color different from the excluded color
    """
    for color in ['red', 'green', 'blue']:
      if color != excluded_color and self.is_consistent(node, color, assignment, csp):
        return True
    return False

# Evaluation

In [11]:
colors = BackTrackingSearch().search(g)

if colors:
    for node, color in sorted(colors.items()):
        print(node, color)
else:
    print("No solution found")

0 red
1 green
2 blue
3 red
4 green
5 red
6 green


# Submission

*   Students download the notebook after completion
*   Rename the notebook in which inserting your student ID at the beginning. \
For example, **123456-lec08-CSPs-HW.ipynb**
*   Finally, submit the file